# Level 1 · Part 2 — Which segmentation can we trust?

The vendor ships one segmentation of this MERSCOPE section, but segmentation is a *choice*,
not ground truth. Here we **re-segment the same ~0.6 mm window four ways** and ask a concrete question:
**how many of the measured transcripts does each method actually place inside a cell, and are those
cells clean?**

We reuse the cropping idiom from Part 1, run three segmenters with **Sopa's own primitives — every
parameter and patch step out in the open, no hidden helpers** — then compare all four segmentations on
two axes:

1. **How much signal is captured** — the fraction of transcripts assigned to a cell.
2. **How clean the cells are** — *negative-marker purity* (do cells co-express genes from
   mutually-exclusive lineages, i.e. did transcripts leak across boundaries?).

## 0. Setup

In [ ]:
import os

# Cellpose reaches through torch, OpenCV, BLAS *and* numba, each of which will otherwise spawn a
# thread per core and thrash the node. We make every worker **single-threaded** (caps below, set
# BEFORE importing torch) and get our parallelism instead by running patches across **Dask worker
# processes** in section 2 -- the pattern Sopa itself uses on a cluster.
for _v in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMBA_NUM_THREADS",
):
    os.environ[_v] = "1"
N_WORKERS = int(os.environ.get("SLURM_CPUS_PER_TASK", "8"))  # one worker per allocated core

import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

import spatialdata as sd
import spatialdata_plot  # noqa: F401  (registers the `.pl` plotting accessor)
import sopa
from spatialdata import get_extent
from spatialdata.models import Image2DModel
from spatialdata.transformations import get_transformation

import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)
import cv2

cv2.setNumThreads(1)

import warnings

warnings.simplefilter("ignore")  # keep the teaching output readable (zarr/dask notices)
sc.settings.verbosity = 1

### Where does the data live — and where do *you* write?

The baseline section is **staged once, read-only, and shared** by everyone on the course — you never
write there. Your crop and its segmentations go in **your clone's git-ignored `data/` folder**, via
the project's `FilePaths` helper, so your outputs stay next to your analysis and never touch the
shared copy (or git). We also point at an independent lineage **reference** (used later, for purity).

In [ ]:
from spatialbrain import FilePaths  # project path helper

# Staged, read-only, shared — NEVER write here.
SHARED_ZARR = (
    "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/"
    "data/wang2025_merfish/processed/UCSF2018-003-MFG_baseline.zarr"
)
# Independent reference of the same tissue with broad lineage labels only (for the purity metric).
SHARED_REF = (
    "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/"
    "data/wang2025_multiome/processed/wang2025_multiome_rna_panel_ref.h5ad"
)

# Your OWN writable copy, in the repo's git-ignored data/ dir.
paths = FilePaths.dataset("wang2025_merfish").create()
CROP_ZARR = paths.processed / "crop_l1.zarr"
print("your crop will be written to:", CROP_ZARR.relative_to(FilePaths.ROOT))

## 1. Make the crop — and meet *in-memory* vs *on-disk*

Same bounding-box crop idiom as Part 1, over a ~0.6 mm window of the same cortical plate. Two teaching points
come together here.

**In-memory vs on-disk.** A freshly cropped SpatialData lives **only in RAM** — `crop.is_backed()` is
`False` and `crop.path` is `None`. Nothing is on disk until you `write()` it. Once written, the object
is **backed**: `is_backed()` is `True` and `crop.path` points at the Zarr store, and big elements
(images, points) become **lazy** — read from that store on demand rather than held whole in RAM.

**Why write at all?** Because the segmenters below **stream image/transcript patches from the Zarr
store** and cache intermediate results next to it. They need the crop *on disk*. So the flow is: crop
→ inspect (in-memory) → `write()` → re-open the backed object → segment.

In [ ]:
sdata = sd.read_zarr(SHARED_ZARR)
image_key = list(sdata.images)[0]
points_key = next(k for k in sdata.points if "patch" not in k.lower())

# microns -> pixels from the transcripts' affine (as in Part 1); 'global' == pixel space.
aff = get_transformation(sdata[points_key], "global").to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))
px_per_um = abs(float(aff[0, 0]))

CX_PX, CY_PX, WIN_UM = 88000, 33000, 600
half = px_per_um * WIN_UM / 2
crop = sdata.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[CX_PX - half, CY_PX - half],
    max_coordinate=[CX_PX + half, CY_PX + half],
    target_coordinate_system="global",
)

print("just cropped -> in memory:")
print("  is_backed():", crop.is_backed(), "| path:", crop.path)

In [ ]:
# Name the two image channels so the segmenters can refer to them (DAPI = nucleus, PolyT = cyto).
img = crop[image_key]
full = img["scale0"]["image"]
crop[image_key] = Image2DModel.parse(
    full.data,
    dims=tuple(full.dims),
    c_coords=["DAPI", "PolyT"],
    scale_factors=[2, 2, 2, 2],
    transformations=get_transformation(img, get_all=True),
)

# Write to YOUR data/ dir. overwrite=True replaces a previous run (no manual shutil.rmtree needed):
# a fresh crop is NOT backed by CROP_ZARR, so overwriting that store is safe.
crop.write(CROP_ZARR, overwrite=True)

print("after write -> on disk:")
print("  is_backed():", crop.is_backed(), "| path:", crop.path)

❓ **Question.** Before `write()`, `crop.path` was `None`; after, it points at your Zarr store.
Why does a *cropped* object start life in-memory even though the *full* section (`sdata`) was backed by
its own store — and what would break if we tried to run the segmenters on the in-memory crop?

In [ ]:
# Re-open the backed crop and work from it.
sdata = sd.read_zarr(CROP_ZARR)
points_key = next(k for k in sdata.points if "patch" not in k.lower())
n_transcripts = int(sdata[points_key].shape[0].compute())
print(f"{n_transcripts:,} transcripts in the {WIN_UM} um crop")
sdata

## 2. Segment four ways — with Sopa primitives, nothing hidden

We already have the **Vendor** segmentation (`authors_cells`). Now we add three more, using Sopa's API
directly so **every parameter and every patch step is visible**:

- **Cellpose** — a deep-learning nuclear/cyto model. It runs on **image patches** (tiles of the image,
  in pixels), because the whole crop is too big to segment at once. `diameter` (pixels) is the key knob:
  at 0.108 µm/px, `diameter=96` ≈ 10 µm, a realistic cell.
- **Baysor** and **Proseg** — transcript-based methods. They run on **transcript patches** (tiles of
  the *points*, in microns) and refine a **prior** segmentation (here Cellpose) using where the mRNA
  actually is.

**Parallelism.** Each patch is segmented in its own **Dask worker process** (`parallelization_backend =
"dask"`), and because we pinned every library to one thread in §0, the workers fill the cores without
oversubscribing them. Sopa prints a progress bar per patch. Expect several minutes on CPU.

In [ ]:
# run one patch per core, each worker single-threaded (threads pinned in section 0)
sopa.settings.parallelization_backend = "dask"
sopa.settings.dask_client_kwargs = {"n_workers": N_WORKERS, "threads_per_worker": 1, "memory_limit": "5GB"}

**Image patches + Cellpose.**

In [ ]:
# tile the image into overlapping patches (pixels); overlap lets a cell on a seam be recovered
sopa.make_image_patches(sdata, patch_width=1500, patch_overlap=120)
print("image patches:", len(sdata["image_patches"]))

# Cellpose on two channels (cytoplasm=PolyT, nucleus=DAPI). Every knob is here in the open:
sopa.segmentation.cellpose(
    sdata,
    channels=["PolyT", "DAPI"],
    diameter=96,  # expected cell diameter in PIXELS (~10 um)
    flow_threshold=2.0,  # cellpose shape tolerance
    cellprob_threshold=-6.0,  # low -> keep dim cells too
    min_area=800,  # drop specks smaller than this (pixels^2)
)
print("Cellpose:", len(sdata["cellpose_boundaries"]), "cells")

**Transcript patches + Baysor** — refine the Cellpose prior with the transcripts.

In [ ]:
sopa.make_transcript_patches(sdata, patch_width=1000, prior_shapes_key="cellpose_boundaries")
sopa.segmentation.baysor(sdata, min_area=10)  # min cell area in MICRONS^2
print("Baysor:", len(sdata["baysor_boundaries"]), "cells")

**Transcript patches + Proseg** — same prior, one patch over the whole crop (Proseg wants all the
transcripts at once).

In [ ]:
sopa.make_transcript_patches(sdata, patch_width=None, prior_shapes_key="cellpose_boundaries")
sopa.segmentation.proseg(sdata)
print("Proseg:", len(sdata["proseg_boundaries"]), "cells")

In [ ]:
# collect the four segmentations by their shapes key
METHODS = {
    "Vendor": "authors_cells",  # the vendor's watershed segmentation
    "CellPose": "cellpose_boundaries",
    "Baysor": "baysor_boundaries",  # Cellpose prior, transcript-refined
    "Proseg": "proseg_boundaries",  # Cellpose prior, transcript-refined
}
{label: (key in sdata.shapes) for label, key in METHODS.items()}

## 3. Look first — boundaries over DAPI

Before any metric, *look*. In a small window, overlay each method's cell outlines on the DAPI nuclei.
Notice how well-separated the cells are here — that will matter for the purity metric later.

🔬 **TASK — Overlay the four segmentations.** Cut a ~150 µm close-up of the crop and, in a 2×2 grid,
draw each method's boundaries (no fill) over DAPI. 💡 **HINT:** `sdata.query.bounding_box(...)` for the
window (as in Part 1), then loop over `METHODS`, chaining `render_images` → `render_shapes(fill_alpha=0,
outline_color=...)` → `.pl.show(ax=...)`.

In [ ]:
ext = get_extent(sdata[image_key], coordinate_system="global")
cx = 0.5 * (ext["x"][0] + ext["x"][1])
cy = 0.5 * (ext["y"][0] + ext["y"][1])
w = px_per_um * 75  # ~150 um
view = sdata.query.bounding_box(
    axes=("x", "y"), min_coordinate=[cx - w, cy - w], max_coordinate=[cx + w, cy + w], target_coordinate_system="global"
)

In [ ]:
# TODO (exercise): 2x2 overlay of each method's boundaries over DAPI on `view`. See the task + hint.
...

## 4. Assign transcripts and apply the *same* QC to every method

For each segmentation we assign transcripts to cells with the **same** Sopa aggregation, then apply the
**same** per-cell QC filter on three covariates:

- **`n_counts`** ≥ 10 transcripts · **`n_genes`** ≥ 3 distinct genes · **`area_um2`** ≥ 15 µm².

Cell **area** needs care: Cellpose polygons are in image **pixels**, the transcript methods in
**microns**, so a raw `.area` isn't comparable. The small helper below rescales every method's area to
**µm²** — defined right here so nothing is hidden.

⚠️ **A caveat on fairness.** Baysor and Proseg already enforce their *own* minimum molecules/area
*during* segmentation, so this shared filter removes very few of their cells, whereas Vendor and
Cellpose lose more. The shared filter makes the *post-hoc* criteria identical, but each method's
built-in floor still differs — keep that in mind when reading the kept-cell counts.

In [ ]:
def shapes_area_um2(sdata, shapes_key, points_key):
    """Cell areas in micron^2, comparable across methods regardless of their intrinsic units."""

    def scale(key):  # element -> pixels scale (abs of the affine's x-diagonal)
        m = get_transformation(sdata[key], "global").to_affine_matrix(input_axes=("x", "y"), output_axes=("x", "y"))
        return abs(float(m[0, 0]))

    micron_per_intrinsic = scale(shapes_key) / scale(points_key)
    return sdata[shapes_key].geometry.area.to_numpy() * micron_per_intrinsic**2

In [ ]:
# TODO (exercise): for each method, aggregate transcripts -> table, compute n_counts / n_genes /
# area_um2, keep cells passing all three thresholds, store the filtered table in `tables[label]` and
# the pre-QC assigned total in `assigned[label]`. See the section text.
...

## 5. Compare the cells — size, counts, genes, and signal captured

How do the four segmentations differ in the *cells* they produce, and how much of the measured signal
does each capture? The **fraction of transcripts assigned** uses the **pre-QC** totals (a transcript is
"assigned" if any cell caught it), so a method isn't penalised for cells that later fail QC.

🔬 **TASK 5.1 — Summarise the four segmentations.** Build a table with, per method: kept cells,
cells/mm², median transcripts and genes per cell, median area (µm²), and the **fraction of transcripts
assigned** (`prop_assigned`). 💡 **HINT:** the pre-QC totals are in `assigned`; `crop_area_mm2` is computed from `WIN_UM`;
`prop_assigned[m] = assigned[m] / n_transcripts`. Define `prop_assigned` here — the next plot reads it.

In [ ]:
# TODO (exercise): build the per-method summary table and define
# prop_assigned[m] = assigned[m] / n_transcripts (the next plot uses it). See task 5.1.
...

In [ ]:
panels = [("transcripts / cell", "n_counts"), ("genes / cell", "n_genes"), ("cell area (um^2)", "area_um2")]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax_, (title, col) in zip(axes, panels, strict=False):
    ax_.violinplot([tables[m].obs[col].to_numpy() for m in METHODS], showmedians=True)
    ax_.set_xticks(range(1, len(METHODS) + 1))
    ax_.set_xticklabels(list(METHODS), rotation=20, ha="right")
    ax_.set(title=title)
fig.tight_layout()

**Signal captured** — the discriminating axis: the fraction of all decoded transcripts that end up
inside a cell. Same transcripts, same crop — only the segmentation differs.

In [ ]:
labels = list(METHODS)
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, [prop_assigned[m] for m in labels], color="steelblue")
ax.set(ylabel="fraction of transcripts assigned", title="Signal captured", ylim=(0, 1))
ax.tick_params(axis="x", rotation=15)
for i, m in enumerate(labels):
    ax.text(i, prop_assigned[m] + 0.02, f"{prop_assigned[m]:.2f}", ha="center")
fig.tight_layout()

## 6. Are the cells *clean*? Negative-marker purity, from the ground up

More transcripts are only good if they land in the *right* cell. How do we check cleanliness with **no
cell-type labels** on the spatial side? The idea (Salas *et al.*; ResolVI): pick pairs of genes from
**mutually-exclusive cell types** — say an oligodendrocyte gene and a glutamatergic-neuron gene. A
correctly segmented cell is one type or the other, so it should almost **never** be positive for both;
when it is, transcripts leaked across a boundary. We'll build that into a score — and, along the way,
watch where its assumptions strain on *developing* tissue.

### 6.1 First, a robust "is this gene ON?" call

Everything below hinges on deciding, per cell, whether a gene is truly expressed. A plain `count > 0`
is treacherous: a **single ambient transcript** — common in both MERFISH and the reference — reads as
"detected". Instead we fit a tiny **2-component Gaussian mixture** on `log1p` counts (a background
component and a signal component) and call the higher one ON. We use this *same* call in both places —
finding exclusive pairs in the reference **and** scoring purity in space — so the two are consistent.

In [ ]:
from sklearn.mixture import GaussianMixture


def gene_is_on(counts, min_positive=20):
    """Boolean 'gene is ON' per cell via a 2-component GMM on log1p counts (fallback: count > 0)."""
    c = np.asarray(counts, float).ravel()
    if (c > 0).sum() < min_positive or c.max() == 0:
        return c > 0
    x = np.log1p(c).reshape(-1, 1)
    gm = GaussianMixture(n_components=2, covariance_type="full", random_state=0, n_init=1).fit(x)
    return gm.predict(x) == int(np.argmax(gm.means_.ravel()))  # higher-mean component = signal


def counts_of(adata, g):
    x = adata[:, g].layers["counts"] if "counts" in adata.layers else adata[:, g].X
    return np.asarray(x.todense()).ravel() if hasattr(x, "todense") else np.asarray(x).ravel()

### 6.2 Mutually-exclusive markers, from *specific* reference populations

Which genes are mutually exclusive? We derive them from the independent **reference** — but with a
choice that matters in *developing* cortex. Broad classes (all "Neuron" vs all "Glia") are too
coarse: progenitors and newborn neurons blur that boundary. So we use **specific terminal cell types**
(`subclass`: glutamatergic & GABAergic neurons, oligodendrocytes, astrocytes, microglia), whose markers
are genuinely type-defining. For each population we keep genes ON in a good fraction of its cells and
almost never elsewhere. *(The per-gene GMM over the reference takes a minute or two.)*

In [ ]:
ref = ad.read_h5ad(SHARED_REF)
genes = list(ref.var_names)
gidx = {g: i for i, g in enumerate(genes)}
Cref = ref.layers["counts"]
Cref = Cref.toarray() if hasattr(Cref, "toarray") else np.asarray(Cref)
labels_ref = ref.obs["subclass"].astype(str).to_numpy()

# per-cell ON/OFF for every gene in the reference (same GMM call we score with)
ON_ref = np.column_stack([gene_is_on(Cref[:, j]) for j in range(len(genes))])

POPULATIONS = ["Glutamatergic neuron", "GABAergic neuron", "Oligodendrocyte", "Astrocyte", "Microglia"]


def population_markers(n_per=8, min_frac_in=0.30, max_frac_out=0.05):
    """Genes specific to each terminal type: ON in >= min_frac_in of its cells, <= max_frac_out elsewhere."""
    out = {}
    for L in POPULATIONS:
        m = labels_ref == L
        frac_in, frac_out = ON_ref[m].mean(0), ON_ref[~m].mean(0)
        ok = (frac_in >= min_frac_in) & (frac_out <= max_frac_out)
        out[L] = [genes[i] for i in np.argsort(frac_in - frac_out)[::-1] if ok[i]][:n_per]
    return out


markers_by_pop = population_markers()
markers_by_pop

In [ ]:
# TODO (exercise): keep cross-type gene pairs whose reference co-ON rate P(both ON)/P(either ON)
# is <= ~0.02, sorted most-exclusive first, in `pairs`. Reuse ON_ref / gidx. See the section text.
...

### 6.3 *See* the exclusivity — reference, then space

Take the most mutually-exclusive pair. Plot every cell's `log1p` counts and colour **red** the cells
that are ON for **both** genes — exactly what the score will count. We show a random sample of the huge
reference so the two panels are comparable, and annotate each with its true **co-detection rate**
`P(both ON) / P(either ON)`. Compare the rates, not the raw red counts.

In [ ]:
ga, gb = pairs[0]
print("most mutually-exclusive pair:", ga, "vs", gb)
rng = np.random.default_rng(0)


def scatter_pair(ax, ca, cb, title, n_show=3000):
    on_a, on_b = gene_is_on(ca), gene_is_on(cb)
    both = on_a & on_b
    either = (on_a | on_b).sum()
    rate = (both.sum() / either) if either else 0.0
    idx = np.arange(ca.size)
    if ca.size > n_show:
        idx = rng.choice(ca.size, n_show, replace=False)
    x, y, bo = np.log1p(ca[idx]), np.log1p(cb[idx]), both[idx]
    ax.scatter((x + rng.normal(0, 0.03, x.size))[~bo], y[~bo], s=5, alpha=0.2, color="lightsteelblue")
    ax.scatter((x + rng.normal(0, 0.03, x.size))[bo], y[bo], s=18, alpha=0.85, color="crimson")
    ax.set(title=f"{title}\nco-detection rate = {rate:.1%}", xlabel=f"log1p {ga}", ylabel=f"log1p {gb}")


vend = tables["Vendor"]
fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))
scatter_pair(axes[0], counts_of(ref, ga), counts_of(ref, gb), "Reference single cells (random sample)")
scatter_pair(axes[1], counts_of(vend, ga), counts_of(vend, gb), "Spatial vendor cells")
fig.tight_layout()

❓ **Question.** The reference co-detection rate is low (the two types really are exclusive), and
the red "ON for both" cells are rare. Is the spatial vendor rate higher or lower? A *higher* spatial
rate means some cells carry both types' transcripts — leakage across a boundary. How would a badly
**over-merging** segmentation change the corner of that cloud?

### 6.4 Turn it into a score

**Negative-marker purity** = `1 − (mean cross-type co-detection rate)` over all pairs — using the same
`gene_is_on` call. Higher = cleaner. A few lines, defined here so nothing is a black box.

In [ ]:
def negative_marker_purity(adata, pairs):
    """1 - mean cross-type double-positive rate over `pairs` (higher = cleaner)."""
    present = {g for pair in pairs for g in pair if g in adata.var_names}
    on = {g: gene_is_on(counts_of(adata, g)) for g in present}
    rates = []
    for ga, gb in pairs:
        if ga in on and gb in on:
            either = (on[ga] | on[gb]).sum()
            if either:
                rates.append((on[ga] & on[gb]).sum() / either)
    return 1.0 - float(np.mean(rates)) if rates else float("nan")

In [ ]:
# TODO (exercise): score negative_marker_purity for each method's table and bar-plot it.
...

❓ **What do you notice?** How does purity compare across the four methods — does it *rank* them,
or is it flat? Use the §3 overlays and the §6.3 scatter to explain why.

### 6.5 A caveat worth teaching — "mutually exclusive" is only *approximate* here

Look again at §6.3: even in the **reference**, a few cells are ON for both markers. The assumption behind
this whole metric — "one cell = one lineage, so these genes never co-occur" — is an **adult-tissue**
idealisation (where Salas *et al.* validated it). It is softer in second-trimester cortex.

❓ **Question.** Give three reasons a cell might be a genuine "double-positive" in the reference
even though we picked the *most* mutually-exclusive pair. (Hint: one technical, one about the markers,
one about the biology of a *developing* brain.)

🚀 **Going further (optional — open-ended).**
- **Which type-pair breaks each method?** Split purity *by pair* and recompute per method. Does a method
  fail on a particular morphology (large astrocytes vs small interneurons)?
- **Is the metric stable across the crop?** Tile the crop, recompute per tile per method, and map it.

## 7. What do the extra transcripts buy us? Compare by eye.

More assigned transcripts mean more counts per cell — but do they translate into cleaner cell-type
structure? Build a UMAP from **each** method's counts and colour it by a few canonical lineage markers,
then judge for yourself.

❓ **QUESTION.** Which method gives the most clearly separated lineages — and is it the one that captured
the *most* transcripts? Does more signal always mean better-resolved structure?

In [ ]:
lineage_of = {"SATB2": "excitatory neurons", "AQP4": "astrocytes", "CLDN5": "vascular"}
marker_genes = [g for g in lineage_of if g in tables["Proseg"].var_names]

embeds = {}
for m in METHODS:
    a = tables[m].copy()
    a.X = a.layers["counts"].copy()
    sc.pp.normalize_total(a)
    sc.pp.log1p(a)
    sc.pp.pca(a, n_comps=30)
    sc.pp.neighbors(a)
    sc.tl.umap(a)
    embeds[m] = a

fig, axes = plt.subplots(len(METHODS), len(marker_genes), figsize=(4 * len(marker_genes), 3.4 * len(METHODS)))
for i, m in enumerate(METHODS):
    for j, g in enumerate(marker_genes):
        sc.pl.umap(
            embeds[m], color=g, ax=axes[i, j], show=False, colorbar_loc=None, title=f"{m}: {g} ({lineage_of[g]})"
        )
fig.tight_layout()

## 8. Conclusion

- **Signal captured** separates the methods: on this crop the **Vendor** segmentation assigns the
  fewest transcripts and all three re-segmentations capture more — **Baysor** the most, then **Proseg**,
  then **CellPose** (still a clear gain over the vendor).
- But *where* those transcripts go differs (see §5 violins): capturing more total signal can mean
  smaller or more numerous cells — there is no free lunch.
- **Purity** is high and flat everywhere — on this well-separated tissue it confirms all segmentations
  are clean but doesn't rank them. It's a **guard rail**, and you built it from the reference up.
- The **UMAPs** let you compare structure across methods directly.

There is no single "correct" segmentation — the right choice trades off total signal, cell count, cell
size, and runtime. The point is to *measure* the trade-off rather than trust the vendor default blindly.

**Further reading.** Sopa ([docs](https://gustaveroussy.github.io/sopa/), Blampey *et al.*, *Nat.
Commun.* 2024) · the purity metric: Salas *et al.* (*Nat. Methods* 2025,
[10.1038/s41592-025-02617-2](https://doi.org/10.1038/s41592-025-02617-2)) and **ResolVI** (Ergen &
Yosef, bioRxiv 2025) · the Cellpose, Baysor, and Proseg papers for the methods themselves.